# Лабораторна робота №5
## Застосування глибинних нейронних мереж у задачах комп'ютерного зору

**Варіант 4** — датасет: класифікація аніме (20 класів)

In [ ]:
# import subprocess, sys

# subprocess.check_call([
#     sys.executable, '-m', 'pip', 'install',
#     'torch', 'torchvision',
#     '--index-url', 'https://download.pytorch.org/whl/cu124',
#     '--force-reinstall'
# ])

## 1. Імпорт бібліотек

In [ ]:
import os
import zipfile
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import StepLR

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True  # прискорює навчання на GPU

## 2. Підготовка датасету

In [ ]:
# Розпакування архіву варіанту
zip_path = "variant-4.zip"  # завантажте файл у робочу директорію або вкажіть шлях
data_root = "./dataset"

if not os.path.exists(data_root):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_root)

# Знаходимо папку з класами (ігноруємо __MACOSX та .DS_Store)
extracted = [d for d in os.listdir(data_root)
             if not d.startswith('__') and not d.startswith('.')]
inner = os.path.join(data_root, extracted[0])
classes = sorted([d for d in os.listdir(inner)
                  if os.path.isdir(os.path.join(inner, d)) and not d.startswith('.')])

print(f"Кількість класів: {len(classes)}")
print("Класи:", classes)

In [ ]:
# Трансформи для власної CNN
transform_cnn = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Трансформи для Transfer Learning (ImageNet mean/std)
transform_tl = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

def make_loaders(transform, test_size=0.2, batch_size=64):
    dataset = ImageFolder(root=inner, transform=transform)
    indices = list(range(len(dataset)))
    train_idx, test_idx = train_test_split(indices, test_size=test_size,
                                           random_state=42, stratify=dataset.targets)
    train_loader = DataLoader(Subset(dataset, train_idx),
                              batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    test_loader  = DataLoader(Subset(dataset, test_idx),
                              batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    return train_loader, test_loader, dataset.classes

train_loader_cnn, test_loader_cnn, class_names = make_loaders(transform_cnn)
train_loader_tl,  test_loader_tl,  _           = make_loaders(transform_tl)

num_classes = len(class_names)
print(f"Train: {len(train_loader_cnn.dataset)}, Test: {len(test_loader_cnn.dataset)}")

## 3. Аналіз прикладу Simple CNN (завдання 2)

In [ ]:
def visualize_images(dataloader, class_names, num_images=5, mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)):
    images, labels = next(iter(dataloader))
    plt.figure(figsize=(15, 3))
    for i in range(num_images):
        img = images[i].numpy().transpose((1, 2, 0))
        img = np.array(std) * img + np.array(mean)
        img = np.clip(img, 0, 1)
        plt.subplot(1, num_images, i + 1)
        plt.imshow(img)
        plt.title(class_names[labels[i]])
        plt.axis('off')
    plt.tight_layout()
    plt.show()

print("Приклади зображень з датасету:")
visualize_images(train_loader_cnn, class_names)

## 4. Розробка власної CNN архітектури (завдання 4)

### 4.1 Архітектура з Flatten

In [ ]:
class CNNWithFlatten(nn.Module):
    """3 згорткові блоки (Conv -> BN -> ReLU -> MaxPool) + Flatten + FC"""
    def __init__(self, num_classes):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # 224 -> 112
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # 112 -> 56
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # 56 -> 28
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x)


class CNNWithGAP(nn.Module):
    """3 згорткові блоки (Conv -> BN -> ReLU -> MaxPool) + Global Average Pooling + FC"""
    def __init__(self, num_classes):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.gap = nn.AdaptiveAvgPool2d(1)   # Global Average Pooling
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.gap(x)
        return self.classifier(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

m_flat = CNNWithFlatten(num_classes)
m_gap  = CNNWithGAP(num_classes)

print(f"Параметри CNNWithFlatten : {count_params(m_flat):,}")
print(f"Параметри CNNWithGAP    : {count_params(m_gap):,}")
print(f"Зменшення               : {count_params(m_flat) / count_params(m_gap):.1f}x")

### 4.2 Навчальний цикл (з Optimizer та Scheduler)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return correct / total, all_preds, all_labels


def train_model(model, train_loader, test_loader, epochs=7, lr=1e-3, label=""):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = StepLR(optimizer, step_size=3, gamma=0.5)

    history = {"loss": [], "acc": [], "val_acc": []}
    for epoch in range(epochs):
        loss, acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, _, _ = evaluate(model, test_loader, device)
        scheduler.step()
        history["loss"].append(loss)
        history["acc"].append(acc)
        history["val_acc"].append(val_acc)
        print(f"[{label}] Epoch {epoch+1}/{epochs}  "
              f"Loss: {loss:.4f}  Acc: {acc:.4f}  Val Acc: {val_acc:.4f}")
    return history

print("Навчання CNNWithGAP...")
model_gap = CNNWithGAP(num_classes)
history_gap = train_model(model_gap, train_loader_cnn, test_loader_cnn, epochs=10, label="CNN+GAP")

## 5. Transfer Learning — ResNet18 (завдання 5)

### 5.1 Feature Extraction (заморожуємо всі шари, крім fc)

In [ ]:
def build_resnet_feature_extraction(num_classes):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

print("Навчання ResNet18 (Feature Extraction)...")
model_fe = build_resnet_feature_extraction(num_classes)
history_fe = train_model(model_fe, train_loader_tl, test_loader_tl, epochs=10, label="FE")

### 5.2 Fine-tuning (розморожуємо layer4 + fc)

In [ ]:
def build_resnet_finetuning(num_classes):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    for name, child in model.named_children():
        if name in ['layer4', 'fc']:
            for param in child.parameters():
                param.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

print("Навчання ResNet18 (Fine-tuning layer4 + fc)...")
model_ft = build_resnet_finetuning(num_classes)
history_ft = train_model(model_ft, train_loader_tl, test_loader_tl, epochs=10, label="FT")

## 6. Аналіз результатів та візуалізація (завдання 6)

### 6.1 Графіки Loss та Accuracy

In [ ]:
epochs = range(1, 11)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs, history_gap["loss"], label="CNN+GAP")
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(epochs, history_gap["val_acc"], label="CNN+GAP")
axes[1].plot(epochs, history_fe["val_acc"],  label="ResNet18 FE")
axes[1].plot(epochs, history_ft["val_acc"],  label="ResNet18 FT")
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Підсумкова таблиця
acc_gap = history_gap["val_acc"][-1]
acc_fe  = history_fe["val_acc"][-1]
acc_ft  = history_ft["val_acc"][-1]
print(f"\nПідсумок (epoch 10):")
print(f"  CNN+GAP       : {acc_gap:.4f}")
print(f"  ResNet18 FE   : {acc_fe:.4f}")
print(f"  ResNet18 FT   : {acc_ft:.4f}")

### 6.2 Confusion Matrix для найкращої моделі

In [ ]:
# Обираємо модель з найвищою val_acc
best_name, best_model, best_loader = max(
    [("CNN+GAP",     model_gap, test_loader_cnn),
     ("ResNet18 FE", model_fe,  test_loader_tl),
     ("ResNet18 FT", model_ft,  test_loader_tl)],
    key=lambda t: evaluate(t[1], t[2], device)[0]
)
print(f"Найкраща модель: {best_name}")

final_acc, all_preds, all_labels = evaluate(best_model, best_loader, device)
print(f"Test Accuracy: {final_acc:.4f}")

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title(f"Confusion Matrix — {best_name}", fontsize=14)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Аналіз найгірших класів
per_class_acc = cm.diagonal() / cm.sum(axis=1)
worst = sorted(zip(class_names, per_class_acc), key=lambda x: x[1])[:5]
print("\nНайгірше розпізнані класи:")
for name, acc in worst:
    print(f"  {name}: {acc:.2f}")

### 6.3 Візуалізація прогнозів

In [ ]:
def visualize_predictions(model, dataloader, device, class_names, num_images=5,
                          mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)):
    model.eval()
    images, labels = next(iter(dataloader))
    with torch.no_grad():
        outputs = model(images.to(device))
        _, predicted = torch.max(outputs, 1)

    plt.figure(figsize=(15, 3))
    for i in range(num_images):
        img = images[i].numpy().transpose((1, 2, 0))
        img = np.array(std) * img + np.array(mean)
        img = np.clip(img, 0, 1)
        color = 'green' if labels[i] == predicted[i] else 'red'
        plt.subplot(1, num_images, i + 1)
        plt.imshow(img)
        plt.title(f"True: {class_names[labels[i]]}\nPred: {class_names[predicted[i]]}",
                  color=color, fontsize=8)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

print(f"Прогнози найкращої моделі ({best_name}):")
mn = (0.485, 0.456, 0.406) if 'ResNet' in best_name else (0.5, 0.5, 0.5)
st = (0.229, 0.224, 0.225) if 'ResNet' in best_name else (0.5, 0.5, 0.5)
visualize_predictions(best_model, best_loader, device, class_names, mean=mn, std=st)